In [1]:
from pathlib import Path
import hashlib
import json
import pandas as pd
from IPython.display import display

In [2]:
TASK = Path.cwd().parent
ROOT = TASK.parent
INPUT, OUTPUT = TASK/'input', TASK/'output'
CONFIG = json.loads((ROOT/'config.json').read_text())
SEED = CONFIG['seed']
PREFIXES = {'arw4':'ARWFour', 'lognormal_grw':'GRW', 'ar1_s':'AROne',
            'hurdle_ar1_s_p':'HurdleAROne', 'ar6_6':'ARSix', 'se_hurdle_s':'SEHurdle'}
FILES = {'annual':'curve_agreement.csv', 'restart':'reentry_agreement.csv', 'paths':'path_agreement.csv',
         'wasserstein':'annual_wasserstein_summary.csv'}

In [3]:
annual = pd.read_csv(INPUT/FILES['annual']).set_index(['source','window','domain','scale','statistic'])
restart = pd.read_csv(INPUT/FILES['restart']).set_index('source')
paths = pd.read_csv(INPUT/FILES['paths']).set_index(['source','window','descriptor'])
w1 = pd.read_csv(INPUT/FILES['wasserstein']).set_index('source')

In [4]:
rows = []
PRECISION = {
    'SEHurdleAnnualWassersteinMean': 2,
    'SEHurdleAnnualWassersteinSlope': 3,
}
def add_macro(prefix, suffix, value, table, selection, column, decimals, scale=1):
    if suffix in ('AnnualWassersteinMean', 'AnnualWassersteinSlope'):
        decimals = 2
    if pd.notna(value):
        rows.append({'macro':prefix+suffix, 'table':table, 'selection':json.dumps(selection),
                     'statistic':column, 'raw_value':value, 'scale':scale,
                     'formatted_value':f'{value*scale:.{decimals}f}'})

for source,prefix in PREFIXES.items():
    for suffix,column in [('AnnualWassersteinMean','mean_wasserstein1'),
                          ('AnnualWassersteinSlope','slope_wasserstein1')]:
        add_macro(prefix,suffix,w1.loc[source,column],'wasserstein',{'source':source},column,4)
    for statistic in ['mean','median','variance']:
        key = (source,'common_generated','annual','raw',statistic)
        fields = ('source','window','domain','scale','statistic')
        add_macro(prefix,'Annual'+statistic.title()+'NRMSEPercent',annual.loc[key,'nrmse'],
                  'annual',dict(zip(fields,key)),'nrmse',2,100)
    add_macro(prefix,'RestartHazardRMSEPoints',restart.loc[source,'risk_weighted_rmse'],
              'restart',{'source':source},'risk_weighted_rmse',2,100)
    key = (source,'common_generated','longest_observed_zero_run')
    add_macro(prefix,'LongestInactiveSpellKSD',paths.loc[key,'ks_D'],'paths',
              dict(zip(('source','window','descriptor'),key)),'ks_D',4)

catalog = pd.DataFrame(rows)

In [5]:
provenance = pd.DataFrame([
    {'table':table, 'source_file':str((INPUT/file).resolve().relative_to(ROOT.parent)),
     'sha256':hashlib.sha256((INPUT/file).read_bytes()).hexdigest()} for table,file in FILES.items()
])
catalog.merge(provenance,on='table').to_csv(OUTPUT/'macro_manifest.csv',index=False)
lines = [
    '% generated by i_report; refresh with make report',
    '% annual raw curves and longest inactive spells use career years 6--20',
    '% annual Wasserstein mean and raw-scale OLS slope use career years 0--20',
    '% restart RMSE is risk-weighted and expressed in percentage points',
    '% Percent commands omit the percent sign'
]
for row in catalog.itertuples():
    lines.append(r'\newcommand{' + chr(92) + row.macro + '}{' + row.formatted_value + r'\xspace}')
temporary = OUTPUT/'trajectory_results.tex.tmp'
temporary.write_text('\n'.join(lines)+'\n')
temporary.replace(OUTPUT/'trajectory_results.tex')
display(catalog[['macro','formatted_value']])

,macro,formatted_value
0,ARWFourAnnualWassersteinMean,0.95
1,ARWFourAnnualWassersteinSlope,0.05
2,ARWFourAnnualMeanNRMSEPercent,7.51
3,ARWFourAnnualMedianNRMSEPercent,8.39
4,ARWFourAnnualVarianceNRMSEPercent,47.74
5,ARWFourLongestInactiveSpellKSD,0.5451
6,GRWAnnualWassersteinMean,3638.20
7,GRWAnnualWassersteinSlope,770.60
8,GRWAnnualMeanNRMSEPercent,112130.91
9,GRWAnnualMedianNRMSEPercent,48.28
